In [ ]:
from bs4 import BeautifulSoup 
import requests
import pandas as pd
from io import StringIO  # Required to fix the Pandas warning
from datetime import datetime, timezone
import os

In [ ]:
# ==============================================================================
# 1. SET WORKING DIRECTORY (Insert this at the top of your script)
# ==============================================================================
target_folder = '/Users/brendonwong/Desktop/GPU power project'

# Create the folder if it doesn't already exist on your Mac
os.makedirs(target_folder, exist_ok=True)

# Change Python's active directory to your target folder
os.chdir(target_folder)
print(f"Active working directory set to: {os.getcwd()}")


# ==============================================================================
# 2. SCRAPE AND SAVE DATA
# ==============================================================================
csv_file = 'verda_pricing_history.csv'
today_date = datetime.now().strftime('%Y-%m-%d')
verda_url = 'https://verda.com/pricing'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

# Check if today's date has already been recorded in the CSV
already_scraped = False

if os.path.exists(csv_file):
    existing_df = pd.read_csv(csv_file)
    if 'Date' in existing_df.columns:
        if today_date in existing_df['Date'].astype(str).values:
            already_scraped = True

# Scrape and save if it's a new date
if not already_scraped:
    print(f"New date detected ({today_date}). Scraping data...")
    
    verda_page = requests.get(verda_url, headers=headers)
    verda_tables = pd.read_html(verda_page.text)
    
    new_df = verda_tables[0].copy()
    
    # Insert 'Date' at position 0 (far left column)
    new_df.insert(0, 'Date', today_date)

    if not os.path.exists(csv_file):
        # Create new CSV with header
        new_df.to_csv(csv_file, index=False, mode='w')
        print(f"Created '{csv_file}' inside '{target_folder}' for {today_date}.")
    else:
        # Append new data rows without repeating headers
        new_df.to_csv(csv_file, index=False, mode='a', header=False)
        print(f"Appended records for {today_date} to '{csv_file}'.")
else:
    print(f"Data for today ({today_date}) already exists in '{csv_file}'. Skipping scrape.")

# Load into your requested coreweave_df / verda_df format
verda_df = [pd.read_csv(csv_file)]
display(verda_df[0])

In [ ]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# 1. Directory & Date Setup
target_folder = '/Users/brendonwong/Desktop/GPU power project'
os.makedirs(target_folder, exist_ok=True)
os.chdir(target_folder)

today_date = datetime.now().strftime('%Y-%m-%d')
na_csv = 'coreweave_na_pricing_history.csv'
eu_csv = 'coreweave_eu_pricing_history.csv'

# 2. Scrape CoreWeave
coreweave_url = 'https://www.coreweave.com/pricing'
coreweave_page = requests.get(coreweave_url)
soup = BeautifulSoup(coreweave_page.text, 'html.parser')

header_container = soup.find('div', class_='table-v2-header')
headers = [cell.text.strip() for cell in header_container.find_all('div', class_='table-v2-cell')]

data = []
for row in soup.find_all('div', class_='table-row-v2'):
    cells = row.find_all('div', class_='table-v2-cell')
    row_values = [" ".join(cell.text.split()) for cell in cells]
    data.append(row_values)

# 3. Insert Date at position 0
df = pd.DataFrame(data, columns=headers)
df.insert(0, 'Date', today_date)

# 4. Split into NA and Europe
north_america_coreweave_df = df.iloc[0:12].copy()
europe_coreweave_df = df.iloc[12:24].copy()

# 5. Check date & append function
def process_csv(df_region, filename):
    already_scraped = False
    if os.path.exists(filename):
        existing_df = pd.read_csv(filename)
        if 'Date' in existing_df.columns and today_date in existing_df['Date'].astype(str).values:
            already_scraped = True

    if not already_scraped:
        if not os.path.exists(filename):
            df_region.to_csv(filename, index=False, mode='w')
            print(f"Created '{filename}' for {today_date}.")
        else:
            df_region.to_csv(filename, index=False, mode='a', header=False)
            print(f"Appended records for {today_date} to '{filename}'.")
    else:
        print(f"Data for {today_date} already exists in '{filename}'. Skipping append.")

# Save both CSVs
process_csv(north_america_coreweave_df, na_csv)
process_csv(europe_coreweave_df, eu_csv)

# Display outputs
display(north_america_coreweave_df)
display(europe_coreweave_df)

In [ ]:
import boto3
import pandas as pd
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
 

# ---------------- Settings ----------------
INSTANCE_FILTERS = ['g*', 'p*', 'inf*', 'trn*', 'vt*']
SPEC_REGION = 'us-east-1'   # region used to list regions and pull specs
DAYS_BACK = 0               # 0 = current price only; up to 90 = price history
 
# Credentials: do NOT hardcode them. Run `aws configure` once, or set the
# AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY environment variables.
session = boto3.Session()
 
 
# ---------------- 1. Regions ----------------
def get_regions():
    ec2 = session.client('ec2', region_name=SPEC_REGION)
    return sorted(r['RegionName'] for r in ec2.describe_regions()['Regions'])
 
 
# ---------------- 2. Instance specs (vCPU, RAM, GPU, network) ----------------
def get_instance_specs(region=SPEC_REGION):
    ec2 = session.client('ec2', region_name=region)
    paginator = ec2.get_paginator('describe_instance_types')
    rows = []
    for page in paginator.paginate(
        Filters=[{'Name': 'instance-type', 'Values': INSTANCE_FILTERS}]
    ):
        for it in page['InstanceTypes']:
            gpu_info = it.get('GpuInfo', {})
            gpus = gpu_info.get('Gpus', [])
            infer = it.get('InferenceAcceleratorInfo', {}).get('Accelerators', [])
            neuron = it.get('NeuronInfo', {}).get('NeuronDevices', [])
 
            # Work out accelerator type/count (GPU, Inferentia or Trainium)
            if gpus:
                accel_name = f"{gpus[0].get('Manufacturer', '')} {gpus[0].get('Name', '')}".strip()
                accel_count = sum(g.get('Count', 0) for g in gpus)
                accel_mem_gib = gpu_info.get('TotalGpuMemoryInMiB', 0) / 1024
            elif neuron:
                accel_name = neuron[0].get('Name')
                accel_count = sum(n.get('Count', 0) for n in neuron)
                accel_mem_gib = it.get('NeuronInfo', {}).get('TotalNeuronDeviceMemoryInMiB', 0) / 1024
            elif infer:
                accel_name = infer[0].get('Name')
                accel_count = sum(a.get('Count', 0) for a in infer)
                accel_mem_gib = sum(
                    a.get('MemoryInfo', {}).get('SizeInMiB', 0) * a.get('Count', 0) for a in infer
                ) / 1024
            else:
                accel_name, accel_count, accel_mem_gib = None, 0, 0
 
            rows.append({
                'InstanceType': it['InstanceType'],
                'vCPU': it['VCpuInfo']['DefaultVCpus'],
                'RAM_GiB': it['MemoryInfo']['SizeInMiB'] / 1024,
                'Accelerator': accel_name,
                'AcceleratorCount': accel_count,
                'AcceleratorMemory_GiB': accel_mem_gib,
                'NetworkPerformance': it['NetworkInfo']['NetworkPerformance'],
                'MaxNetworkCards': it['NetworkInfo'].get('MaximumNetworkCards'),
                'InstanceStorage_GB': it.get('InstanceStorageInfo', {}).get('TotalSizeInGB', 0),
                'CPUArch': ','.join(it['ProcessorInfo']['SupportedArchitectures']),
            })
    return pd.DataFrame(rows)
 
 
# ---------------- 3. Spot prices across regions ----------------
def get_spot_prices(region):
    ec2 = session.client('ec2', region_name=region)
    start = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)
    paginator = ec2.get_paginator('describe_spot_price_history')
    rows = []
    for page in paginator.paginate(
        Filters=[{'Name': 'instance-type', 'Values': INSTANCE_FILTERS}],
        ProductDescriptions=['Linux/UNIX', 'Windows'],
        StartTime=start,
    ):
        for item in page['SpotPriceHistory']:
            rows.append({
                'Region': region,
                'InstanceType': item['InstanceType'],
                'OS': 'Linux' if item['ProductDescription'] == 'Linux/UNIX' else 'Windows',
                'AvailabilityZone': item['AvailabilityZone'],
                'SpotPrice': float(item['SpotPrice']),
                'Timestamp': item['Timestamp'],
            })
    return rows
 
 
regions = get_regions()
all_rows = []
extracted_at = datetime.now(ZoneInfo('Asia/Singapore'))
for region in regions:
    try:
        rows = get_spot_prices(region)
        all_rows.extend(rows)
        print(f"{region}: {len(rows)} records")
    except Exception as e:
        print(f"{region}: skipped ({type(e).__name__}: {e})")
 
history_df = pd.DataFrame(all_rows)
 
# Latest price per Region / Zone / OS / Instance Type
snapshot_df = (
    history_df.sort_values('Timestamp', ascending=False)
    .drop_duplicates(subset=['Region', 'InstanceType', 'OS', 'AvailabilityZone'])
    .reset_index(drop=True)
)
 
# ---------------- 4. Merge with specs ----------------
specs_df = get_instance_specs()
df = snapshot_df.merge(specs_df, on='InstanceType', how='left')
df['ExtractionDate'] = extracted_at.date()
df['ExtractedAt'] = extracted_at.strftime('%Y-%m-%d %H:%M:%S')
df = df.rename(columns={'Timestamp': 'PriceLastChanged'})
 
# Normalised price per accelerator-hour (useful for comparing across chips)
df['SpotPricePerAccelerator'] = df['SpotPrice'] / df['AcceleratorCount'].where(df['AcceleratorCount'] > 0)

AWS_df = [df] 
 
# ---------------- 5. Append to CSV ----------------
AMZN_csv = 'AMZN.csv'

def append_to_csv(new_df, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        old = pd.read_csv(path)
        # If you re-run on the same day, replace that day's rows instead of duplicating them
        old = old[old['ExtractionDate'] != new_df['ExtractionDate'].iloc[0]]
        combined = pd.concat([old, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(path, index=False)
    print(f"Saved {len(new_df)} rows for {new_df['ExtractionDate'].iloc[0]} -> {path} ({len(combined)} rows total)")

append_to_csv(AWS_df[0], AMZN_csv)
display(AWS_df[0])